# Lecture 8.4: Connecting a Local MCP Server (stdio)

**Section 08: MCP, Connecting External Systems**

Pixel Forge already has a bug tracker. It is a small Python program that speaks MCP over standard input and output, which is called a **stdio server**. In this notebook you do not write any tools. You write a short config that tells the Claude Agent SDK how to start that server, and the SDK does the rest: it launches the server as its own process, connects to it, and hands its tools to the agent.

By the end of this notebook you will have:

- created the studio's bug database and a direct way to read it (`show_bugs()`),
- written the ready-made server file to disk,
- configured it as a stdio server with `command`, `args` and `env`,
- seen its four tools appear in the init message as `mcp__bugs__...`,
- proved the server runs as a separate process,
- used it to read bugs, report one, close one, and handle an error, checking each change against the database itself.

## Cell 1: Install the Claude Agent SDK and the MCP package

This notebook needs two packages:

| Package | Why this notebook needs it |
|---|---|
| `claude-agent-sdk` | Runs the agent. It also bundles the Claude Code engine that starts and talks to MCP servers. |
| `mcp` | The official Model Context Protocol package. The bug tracker server in Cell 5 is built with it. |

Both versions are pinned so every example runs exactly as shown in the videos.

**Why `mcp` is pinned as well.** The SDK itself already depends on `mcp`, and it accepts any version from 1.23 up to (but not including) 3. That range spans two major versions with different APIs. The server file in this notebook uses the **mcp 2.x** API (`MCPServer`), so we pin `mcp==2.3.0` to make sure the server code and the installed package match.

If the packages are already installed in the current Colab session, pip simply confirms them and finishes quickly. The `-q` flag keeps the output short.

In [ ]:
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
# The mcp package is pinned too, because bug_server.py uses the mcp 2.x API.
!pip install claude-agent-sdk==0.2.188 mcp==2.3.0 -q

## Cell 2: Load your API key from Colab Secrets

The agent needs your Anthropic API key. We read it from Colab Secrets so it never appears in the notebook.

**How to add the secret (one time per Google account):**

1. Click the **key icon** in Colab's left sidebar.
2. Click **Add new secret**.
3. Name it exactly `ANTHROPIC_API_KEY`.
4. Paste your key as the value.
5. Switch **Notebook access** on for this notebook.

The cell copies the key into the environment variable `ANTHROPIC_API_KEY`, which is where the SDK looks for it.

> **Running locally instead of Colab?** Skip this cell and set the variable in your terminal before starting Jupyter, for example `export ANTHROPIC_API_KEY=...` on macOS or Linux.

In [ ]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

`MODEL_NAME` is the single place the model is set. Every `ClaudeAgentOptions` in this notebook uses `model=MODEL_NAME`, so changing this one line changes the model across the whole notebook.

We use Claude Haiku 5.5. For other available models, follow the link in the comment, since model names change with new releases.

In [ ]:
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-5-5"

## Cell 4: Create Pixel Forge's bug database

Pixel Forge, Maya and Arjun's two-person game studio, tracks bugs in a small SQLite file at `/content/pixelforge/bugs.db`. This cell creates that file with ten game bugs.

The table has four columns:

| Column | Meaning | Example |
|---|---|---|
| `id` | Bug ID, `B-1` to `B-10` | `B-4` |
| `title` | Short description | `Coin counter shows negative values after a shop refund` |
| `severity` | `low`, `medium`, `high` or `critical` | `critical` |
| `status` | `open` or `closed` | `open` |

A few facts to keep in mind, because later cells test them:

- Exactly **two** bugs are critical **and** still open: `B-2` and `B-5`. (`B-10` is critical but already closed.)
- `B-4` is open and medium severity.
- Three bugs are closed.

The cell also defines `show_bugs()`. It reads the file directly with `sqlite3` and prints every row. It does not go through the agent or the server, so it is our **ground truth**: whenever the agent says it changed something, `show_bugs()` shows what is really on disk.

The cell drops and recreates the table each time it runs, so rerunning it resets the data to the starting ten bugs.

In [ ]:
import os
import sqlite3

os.makedirs("/content/pixelforge", exist_ok=True)
DB_PATH = "/content/pixelforge/bugs.db"

bugs = [
    ("B-1", "Player sprite flickers while wall-jumping", "medium", "open"),
    ("B-2", "Save file corrupts after the lava boss fight", "critical", "open"),
    ("B-3", "Pause menu music keeps playing after resume", "low", "closed"),
    ("B-4", "Coin counter shows negative values after a shop refund", "medium", "open"),
    ("B-5", "Game crashes when a controller disconnects mid-level", "critical", "open"),
    ("B-6", "Shop ignores the discount potion", "low", "open"),
    ("B-7", "Slime enemy gets stuck inside the waterfall wall", "high", "open"),
    ("B-8", "Credits text overlaps the studio logo", "low", "closed"),
    ("B-9", "Double jump fails on moving platforms", "high", "closed"),
    ("B-10", "Checkpoint flag resets after respawn", "critical", "closed"),
]

with sqlite3.connect(DB_PATH) as conn:
    conn.execute("DROP TABLE IF EXISTS bugs")
    conn.execute(
        "CREATE TABLE bugs (id TEXT PRIMARY KEY, title TEXT, severity TEXT, status TEXT)"
    )
    conn.executemany("INSERT INTO bugs VALUES (?, ?, ?, ?)", bugs)


def show_bugs():
    """Print every bug straight from the database file."""
    with sqlite3.connect(DB_PATH) as conn:
        rows = conn.execute(
            "SELECT id, severity, status, title FROM bugs "
            "ORDER BY CAST(SUBSTR(id, 3) AS INTEGER)"
        ).fetchall()
    print(f"{'ID':<6}{'SEVERITY':<10}{'STATUS':<8}TITLE")
    for bug_id, severity, status, title in rows:
        print(f"{bug_id:<6}{severity:<10}{status:<8}{title}")
    print(f"\n{len(rows)} bugs in {DB_PATH}")


show_bugs()

## Cell 5: Write the studio's bug tracker server to disk

This is Pixel Forge's ready-made bug tracker server. You do not need to follow every line. The point of this lecture is **connecting** to a server someone else wrote, not writing one. This cell just saves it as `/content/pixelforge/bug_server.py`, then prints its length and the four tool names it defines.

A few things worth knowing about the file:

- **It is a normal Python script.** It is never imported by the notebook. The SDK will start it as a separate program and talk to it through standard input and output.
- **Each `@server.tool()` function becomes an MCP tool.** The SDK reads the tool list from the running server.
- **Type hints become the input schema.** `Literal["low", "medium", "high", "critical"]` becomes a fixed list of allowed values the agent must choose from.
- **Docstrings become the tool descriptions** that the agent reads when deciding which tool to call.
- **It reads its database path from an environment variable**, `BUGS_DB`, with a default. Cell 6 sets it.
- **Expected failures raise `ToolError`.** In mcp 2.x, a `ToolError` sends its message back to the agent. A plain Python exception such as `ValueError` reaches the agent only as a generic "Error executing tool" message, with the reason hidden.

| Tool | What it does |
|---|---|
| `list_bugs` | Lists bugs filtered by status: `open`, `closed` or `all` |
| `report_bug` | Adds a new open bug with a title and severity |
| `close_bug` | Closes an open bug by ID, or raises `ToolError` if there is no such open bug |
| `server_info` | Reports the server's own process ID and database path |

> **A note on older examples.** Many MCP examples online still use `from mcp.server.fastmcp import FastMCP`. In mcp 2.x, `FastMCP` was renamed to `MCPServer`, and the old import raises `ModuleNotFoundError`. If you copy an older example, change the import.

The last line uses a shell `grep` to pull the function names that follow each `@server.tool()` decorator.

In [ ]:
SERVER_CODE = r'''"""Pixel Forge bug tracker: a small MCP server over stdio.

The agent never imports this file. The SDK starts it as a separate
process and talks to it through standard input and output.
"""
import os
import sqlite3
from typing import Literal

from mcp.server.mcpserver import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

DB_PATH = os.environ.get("BUGS_DB", "/content/pixelforge/bugs.db")

server = MCPServer("bugs")


def _connect():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


@server.tool()
def list_bugs(status: Literal["open", "closed", "all"] = "open") -> str:
    """List bugs in the tracker, filtered by status (open, closed or all)."""
    with _connect() as conn:
        if status == "all":
            rows = conn.execute("SELECT * FROM bugs ORDER BY id").fetchall()
        else:
            rows = conn.execute(
                "SELECT * FROM bugs WHERE status = ? ORDER BY id", (status,)
            ).fetchall()
    if not rows:
        return f"No {status} bugs."
    return "\n".join(
        f"{r['id']} | {r['severity']} | {r['status']} | {r['title']}" for r in rows
    )


@server.tool()
def report_bug(title: str, severity: Literal["low", "medium", "high", "critical"]) -> str:
    """Report a new bug. It starts with status open."""
    with _connect() as conn:
        count = conn.execute("SELECT COUNT(*) FROM bugs").fetchone()[0]
        bug_id = f"B-{count + 1}"
        conn.execute(
            "INSERT INTO bugs (id, title, severity, status) VALUES (?, ?, ?, 'open')",
            (bug_id, title, severity),
        )
    return f"Reported {bug_id}: {title} ({severity})"


@server.tool()
def close_bug(bug_id: str) -> str:
    """Close an open bug by its ID, for example B-4."""
    with _connect() as conn:
        cur = conn.execute(
            "UPDATE bugs SET status = 'closed' WHERE id = ? AND status = 'open'",
            (bug_id,),
        )
    if cur.rowcount == 0:
        raise ToolError(f"No open bug with ID {bug_id}.")
    return f"Closed {bug_id}"


@server.tool()
def server_info() -> str:
    """Report this server's process ID and database path."""
    return f"pid={os.getpid()} db={DB_PATH}"


if __name__ == "__main__":
    server.run()  # stdio is the default transport
'''

SERVER_PATH = "/content/pixelforge/bug_server.py"
with open(SERVER_PATH, "w") as f:
    f.write(SERVER_CODE)

print(SERVER_PATH, "written:", len(SERVER_CODE.splitlines()), "lines")
print("Tools defined in the file:")
!grep -A1 "@server.tool()" {SERVER_PATH} | grep -o "def [a-z_]*"

## Cell 6: Describe how to start the server (the stdio config)

This is the core of the lecture. You are **not** starting the server here. You are writing a recipe that tells the SDK **how** to start it. The SDK starts it when an agent run begins.

A stdio server config is a plain dictionary with three fields:

| Field | Value here | Meaning |
|---|---|---|
| `command` | `sys.executable` | The program to run. `sys.executable` is the path of the notebook's own Python, which already has `mcp` installed from Cell 1. |
| `args` | `["/content/pixelforge/bug_server.py"]` | Arguments passed to that program. Here, the script to run. |
| `env` | `{"BUGS_DB": "/content/pixelforge/bugs.db"}` | Extra environment variables for the server process. |

There is also an optional `"type": "stdio"` field. It is the default, so we leave it out.

**`env` adds settings. It does not replace the environment.** In our test, the server received `BUGS_DB` from the config and also inherited ordinary variables such as `PATH` and `HOME`. It did **not** receive `ANTHROPIC_API_KEY`. That last point is what we observed in our test, not a documented guarantee, so pass a server only the settings it needs through `env`.

**The server name sets the tool names.** The key in `mcp_servers` is `"bugs"`, so every tool from this server is named `mcp__bugs__<tool>`, for example `mcp__bugs__close_bug`.

**Least privilege.** `allowed_tools=["mcp__bugs__*"]` approves every tool from the `bugs` server and nothing else. No built-in tools such as `Bash` or `Write` are approved.

**Server lifecycle.** You never start or stop the server yourself. For each run, the SDK starts the server process when the run begins and stops it when the run ends.

`OPTIONS` is kept as a plain `dict` so each run can build a fresh `ClaudeAgentOptions(**OPTIONS)`.

The `run(prompt)` helper runs one query and prints three kinds of lines as they stream in:

| Printed line | Comes from |
|---|---|
| `TOOL CALL` | A `ToolUseBlock` inside an `AssistantMessage` |
| `TOOL RESULT` or `TOOL ERROR` | A `ToolResultBlock` inside a `UserMessage` (`is_error` picks the label) |
| `FINAL ANSWER` | The `ResultMessage` at the end of the stream |

It also returns every message so later cells can inspect them. Running this cell prints nothing.

In [ ]:
# LIVE CODE - type during recording

## Cell 7: Watch the server's tools appear

Before doing real work, let's confirm the SDK started the server and found its tools.

At the start of every run, the stream begins with a `SystemMessage` whose `subtype` is `"init"`. Its `data` includes:

- `mcp_servers`: each configured server with its connection `status` (`connected`, `pending`, `failed`, `needs-auth` or `disabled`),
- `tools`: every tool the agent can see. MCP tools start with `mcp__`.

For a stdio server passed in `mcp_servers`, the first turn waits for the server to connect (up to 30 seconds by default), so the init message should show `bugs: connected` and four `mcp__bugs__...` tools. A server that has not connected by the time the init message is sent is missing from the `tools` list.

The prompt only asks for one word, so the agent has no reason to call any tool. We just read the init message.

> The SDK also has a method, `get_mcp_status()` on `ClaudeSDKClient`, that reports live server status at any point in a session. Here we use the init message, which every run already provides.

In [ ]:
from claude_agent_sdk import SystemMessage

async for message in query(
    prompt="Reply with the single word: ready.",
    options=ClaudeAgentOptions(**OPTIONS),
):
    if isinstance(message, SystemMessage) and message.subtype == "init":
        for server in message.data.get("mcp_servers", []):
            print(f"Server {server['name']}: {server['status']}")
        mcp_tools = [t for t in message.data.get("tools", []) if t.startswith("mcp__")]
        print("MCP tools in the init message:")
        for name in mcp_tools:
            print("   ", name)
    elif isinstance(message, ResultMessage):
        print("Agent says:", message.result)

## Cell 8: Prove the server is a separate process

Every running program gets a **process ID** (PID) from the operating system. The `server_info` tool returns the server's PID. This cell asks the agent to call it, then prints the notebook's own PID with `os.getpid()`.

If the two numbers differ, the server is a different program from your notebook.

The full chain looks like this:

| Process | Started by |
|---|---|
| Your notebook (Python) | Colab |
| The Claude Code engine bundled with the SDK | The SDK, when the run starts |
| `bug_server.py` | The Claude Code engine, using your `command` and `args` |

So the server's parent process is the agent's engine, not the notebook. The exact PID numbers will be different on every run and every machine. What matters is that the two numbers are not the same.

In [ ]:
# LIVE CODE - type during recording

## Cell 9: Read from the tracker

Now the agent does real work. We ask which critical bugs are still open. The agent decides which tool to call. Most likely it calls `list_bugs`, then filters for critical ones.

After the run, the cell queries `bugs.db` directly with SQL. From Cell 4 we know the answer must be **B-2** and **B-5**.

You may also see a `ToolSearch` call before the MCP tool. The engine can use it to load a tool's full definition. The wording of the final answer varies between runs. The bug IDs should not.

In [ ]:
messages = await run("Which critical bugs are still open?")

with sqlite3.connect(DB_PATH) as conn:
    truth = conn.execute(
        "SELECT id, title FROM bugs WHERE severity = 'critical' AND status = 'open'"
    ).fetchall()

print("\nGround truth from bugs.db:")
for bug_id, title in truth:
    print(f"   {bug_id}  {title}")

## Cell 10: Change the tracker

Next, the agent reports a new bug. The agent should call `report_bug` with a title and severity `high`. The server gives it the next free ID, which is `B-11` on a fresh database.

`show_bugs()` then reads the file directly. The new row should appear with status `open`.

The exact title text may vary, because the agent writes it from the prompt. The new row on disk is the proof that the change happened.

In [ ]:
messages = await run(
    "Report a new bug: the player falls through the floor on level 3. Severity high."
)

print()
show_bugs()

## Cell 11: Close a bug, and handle an error

One prompt, two closes:

- `B-4` is open, so closing it should succeed with `Closed B-4`.
- `B-99` does not exist, so `close_bug` raises `ToolError("No open bug with ID B-99.")`.

Because the server raised `ToolError`, that message travels back to the agent as an error result, and the agent can explain what went wrong.

After the run, the cell pairs each `close_bug` call with its result (matching `tool_use_id`) and prints `is_error` and the content for each. Then `show_bugs()` confirms on disk that **B-4** is closed and nothing else changed.

The agent may make the two calls in one step or one after the other, and its final wording varies. The two results and the database should not.

In [ ]:
messages = await run("Close bug B-4, then close bug B-99.")

# Pair each close_bug call with its result
calls = {}
for message in messages:
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, ToolUseBlock) and block.name == "mcp__bugs__close_bug":
                calls[block.id] = block.input.get("bug_id")
    elif isinstance(message, UserMessage) and isinstance(message.content, list):
        for block in message.content:
            if isinstance(block, ToolResultBlock) and block.tool_use_id in calls:
                print(f"\n{calls[block.tool_use_id]}: is_error={block.is_error}")
                print("   ", block.content)

print()
show_bugs()

## Wrap-up

- A stdio config is a recipe: `command`, `args` and `env`.
- The SDK starts the server as a separate process for each run, and stops it when the run ends.
- `env` adds settings on top of the inherited environment.
- MCP tools appear as `mcp__<server>__<tool>` and are approved with `allowed_tools`, just like in-process tools.
- You can use a server without writing any of its tools. Your job is the config.

**Next lecture:** connecting to a remote MCP server over HTTP, where the server is not on this machine at all.